In [1]:
# Chapter 1
from ddgs import DDGS

from fastcore.all import *

def search_images(keywords, max_images=200): return L(DDGS()
    .images(keywords, max_results=max_images)).itemgot('image')
import time,json

urls = search_images('bird photos', max_images=1)
urls[0]




DDGSException: DecodeError: DecodeError('Body collection error: error decoding response body')

In [ ]:
from fastdownload import download_url
dest = 'bird.jpg'
download_url(urls[0], dest, show_progress=True)

from fastai.vision.all import *
im = Image.open(dest)
im.to_thumb(256,256)

In [ ]:
download_url(search_images('forest photos', max_images=1)[0], 
             'forest.jpg', show_progress=False)
Image.open('forest.jpg').to_thumb(256,256)

In [ ]:
searches = 'forest', 'bird'
path = Path('bird_or_not')

for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo'))
    time.sleep(5)
    resize_images(path/o, max_size=400, dest=path/o)
    
    

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)
dls.show_batch(max_n=6)

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(10)

In [ ]:
is_item, _, probs = learn.predict(PILImage.create('forest.jpg'))
print(f'this is a: {is_item}')
print(f'probability it is a {is_item}: {probs[0]}')

# PILImage.create('bird.jpg')

In [ ]:
from fastai.vision.all import *
interp = ClassificationInterpretation.from_learner(learn)
interp